# Phase 5 Reporting Figures: XAI vs Lesion Masks and Pseudo-Concepts

This notebook reads the Phase 5 CSV result files from `outputs/{RUN_NAME}/metrics/` and generates supervisor/report-ready tables and figures.

It is designed to support the report narrative:

1. Lesion-level XAI evaluation
2. Pseudo-concept-level XAI evaluation
3. Lesion-size sensitivity
4. Metric-behaviour comparison, especially Dice vs TIxAI
5. Melanoma vs non-melanoma stratified analysis, if label metadata is available
6. Chance-adjusted pseudo-concept evaluation, if chance-adjusted CSVs are available

Run this notebook from the project root:

`/home/tbeliere/Projects/DCU/2026-mcm-XAI-skin-lesion`


In [ ]:

# 1. Imports and paths


from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Notebook is located in src/, so project root is one level above.
ROOT = Path("..").resolve()

OUTPUTS_DIR = ROOT / "outputs"
RUN_NAME = "full_dataset" # change run_name to match the run you want to analyse

PHASE5_DIR = OUTPUTS_DIR / f"phase5_{RUN_NAME}"

METRICS_DIR = PHASE5_DIR / "metrics"
CHANCE_DIR = METRICS_DIR / "chance_adjusted"

FIG_DIR = PHASE5_DIR / "figures" / "reporting"
TABLE_DIR = PHASE5_DIR / "tables" / "reporting"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("METRICS_DIR:", METRICS_DIR)
print("CHANCE_DIR:", CHANCE_DIR)
print("FIG_DIR:", FIG_DIR)
print("TABLE_DIR:", TABLE_DIR)

In [ ]:

# 2. Load CSV files


def read_csv_if_exists(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        print(f"Missing: {path}")
        return None
    try:
        df = pd.read_csv(path)
        print(f"Loaded {path.name}: {df.shape}")
        return df
    except pd.errors.EmptyDataError:
        print(f"Empty file: {path.name}")
        return pd.DataFrame()

metrics_path = METRICS_DIR / f"phase5_{RUN_NAME}_xai_concept_metrics.csv"
mask_summary_path = METRICS_DIR / f"phase5_{RUN_NAME}_mask_summary_by_method.csv"
method_concept_summary_path = METRICS_DIR / f"phase5_{RUN_NAME}_summary_by_method_concept.csv"
size_summary_path = METRICS_DIR / f"phase5_{RUN_NAME}_summary_by_size_class.csv"
dataset_summary_path = METRICS_DIR / f"phase5_{RUN_NAME}_summary_by_dataset.csv"
manifest_path = METRICS_DIR / f"phase5_{RUN_NAME}_resolved_manifest.csv"
# manifest_path = OUTPUTS_DIR / RUN_NAME / "manifests" / f"{RUN_NAME}_xai_manifest.csv"

chance_metrics_path = CHANCE_DIR / f"phase5_{RUN_NAME}_ chance_adjusted_xai_concept_metrics.csv"
chance_method_concept_path = CHANCE_DIR / f"phase5_{RUN_NAME}_ chance_summary_by_method_concept.csv"
chance_label_path = CHANCE_DIR / f"phase5_{RUN_NAME}_ chance_summary_by_label.csv"
chance_size_path = CHANCE_DIR / f"phase5_{RUN_NAME}_ chance_summary_by_size_class.csv"

metrics_df = read_csv_if_exists(metrics_path)
mask_summary = read_csv_if_exists(mask_summary_path)
method_concept_summary = read_csv_if_exists(method_concept_summary_path)
size_summary = read_csv_if_exists(size_summary_path)
dataset_summary = read_csv_if_exists(dataset_summary_path)
manifest = read_csv_if_exists(manifest_path)

chance_metrics = read_csv_if_exists(chance_metrics_path)
chance_method_concept = read_csv_if_exists(chance_method_concept_path)
chance_label = read_csv_if_exists(chance_label_path)
chance_size = read_csv_if_exists(chance_size_path)


In [ ]:

# 3. Helper functions


METHOD_ORDER = ["gradcam", "shap", "lime"]
METHOD_LABELS = {
    "gradcam": "Grad-CAM",
    "shap": "SHAP",
    "lime": "LIME",
}

SIZE_ORDER = ["normal", "small", "tiny", "very_tiny", "border_touching"]

CORE_CONCEPTS = [
    "asymmetry",
    "border_default",
    "border_w8_dil2_sigma5",
    "border_w12_dil4_sigma8",
    "border_w16_dil6_sigma10",
    "border_w16_dil6_sigma10_dist",
    "colour_heterogeneity",
]

CONCEPT_LABELS = {
    "asymmetry": "Asymmetry",
    "border_default": "Border default",
    "border_w8_dil2_sigma5": "Border w8 d2 s5",
    "border_w12_dil4_sigma8": "Border w12 d4 s8",
    "border_w16_dil6_sigma10": "Border w16 d6 s10",
    "border_w16_dil6_sigma10_dist": "Border dist.",
    "colour_heterogeneity": "Colour heterogeneity",
}

def save_current_fig(name: str, dpi: int = 200):
    path = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"Saved: {path}")
    plt.show()

def method_display(series):
    return series.map(METHOD_LABELS).fillna(series)

def concept_display(series):
    return series.map(CONCEPT_LABELS).fillna(series)

def export_table(df: pd.DataFrame, name: str):
    path = TABLE_DIR / name
    df.to_csv(path, index=False)
    print(f"Saved: {path}")
    return df

def pivot_for_heatmap(df, index_col, column_col, value_col):
    out = df.pivot_table(index=index_col, columns=column_col, values=value_col, aggfunc="mean")
    return out

def add_value_labels(ax, fmt="{:.3f}", rotation=0):
    for container in ax.containers:
        ax.bar_label(container, fmt=fmt, fontsize=8, rotation=rotation)


## 4. Lesion-Level XAI Results

Purpose: show the standard baseline evaluation first.

This answers:

> Do Grad-CAM, SHAP and LIME focus on the lesion region at all?

The table and plot below come from `phase5_{RUN_NAME}_mask_summary_by_method.csv`.


In [ ]:

# 4A. Lesion-level summary table


if mask_summary is not None and not mask_summary.empty:
    lesion_table = (
        mask_summary
        .copy()
        .assign(method=lambda d: method_display(d["xai_method"]))
        [["method", "n", "mean_iou", "mean_dice", "mean_sir", "mean_tixai", "mean_mask_area", "mean_xai_area"]]
        .sort_values("method")
    )

    display(lesion_table.round(4))
    export_table(lesion_table.round(6), "table_lesion_level_xai_summary.csv")
else:
    print("mask_summary is unavailable.")


In [ ]:

# 4B. Figure: lesion-level metric comparison


if mask_summary is not None and not mask_summary.empty:
    plot_df = mask_summary.copy()
    plot_df["method"] = method_display(plot_df["xai_method"])

    metric_cols = ["mean_iou", "mean_dice", "mean_sir", "mean_tixai"]
    long_df = plot_df.melt(
        id_vars=["method"],
        value_vars=metric_cols,
        var_name="metric",
        value_name="value"
    )
    long_df["metric"] = long_df["metric"].str.replace("mean_", "", regex=False).str.upper()

    pivot = long_df.pivot(index="metric", columns="method", values="value")
    pivot = pivot[[METHOD_LABELS[m] for m in METHOD_ORDER if METHOD_LABELS[m] in pivot.columns]]

    ax = pivot.plot(kind="bar", figsize=(9, 5), rot=0)
    ax.set_title("Lesion-level XAI localisation metrics")
    ax.set_xlabel("Metric")
    ax.set_ylabel("Mean score")
    ax.legend(title="XAI method")
    ax.grid(axis="y", alpha=0.3)

    save_current_fig("fig_lesion_level_metric_comparison.png")
else:
    print("mask_summary is unavailable.")


## 5. Pseudo-Concept-Level Results

Purpose: compare each XAI method against the engineered pseudo-concept maps.

This answers:

> Which XAI methods align most strongly with asymmetry, border irregularity and colour heterogeneity proxies?


In [ ]:
# ============================================================
# 5A. Pseudo-concept summary table
# ============================================================

if method_concept_summary is not None and not method_concept_summary.empty:
    concept_table = (
        method_concept_summary
        .copy()
        .assign(
            method=lambda d: method_display(d["xai_method"]),
            concept_label=lambda d: concept_display(d["concept"])
        )
        [["concept_label", "method", "n", "mean_iou", "mean_dice", "mean_sir", "mean_tixai", "mean_pearson_corr", "mean_concept_area"]]
        .sort_values(["concept_label", "method"])
    )

    display(concept_table.round(4))
    export_table(concept_table.round(6), "table_pseudo_concept_xai_summary.csv")
else:
    print("method_concept_summary is unavailable.")


In [ ]:
# ============================================================
# 5B. Figure: pseudo-concept Dice heatmap
# ============================================================

if method_concept_summary is not None and not method_concept_summary.empty:
    heat_df = method_concept_summary.copy()
    heat_df["method"] = method_display(heat_df["xai_method"])
    heat_df["concept_label"] = concept_display(heat_df["concept"])

    heat = pivot_for_heatmap(heat_df, "concept_label", "method", "mean_dice")
    ordered_concepts = [CONCEPT_LABELS[c] for c in CORE_CONCEPTS if CONCEPT_LABELS[c] in heat.index]
    ordered_methods = [METHOD_LABELS[m] for m in METHOD_ORDER if METHOD_LABELS[m] in heat.columns]
    heat = heat.loc[ordered_concepts, ordered_methods]

    fig, ax = plt.subplots(figsize=(8, 5))
    im = ax.imshow(heat.values, aspect="auto")

    ax.set_xticks(np.arange(len(heat.columns)))
    ax.set_xticklabels(heat.columns)
    ax.set_yticks(np.arange(len(heat.index)))
    ax.set_yticklabels(heat.index)

    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            ax.text(j, i, f"{heat.iloc[i, j]:.3f}", ha="center", va="center", fontsize=8)

    ax.set_title("Pseudo-concept alignment by method: Dice")
    fig.colorbar(im, ax=ax, label="Mean Dice")
    save_current_fig("fig_pseudo_concept_dice_heatmap.png")
else:
    print("method_concept_summary is unavailable.")


In [ ]:
# ============================================================
# 5C. Figure: pseudo-concept TIxAI heatmap
# ============================================================

if method_concept_summary is not None and not method_concept_summary.empty:
    heat_df = method_concept_summary.copy()
    heat_df["method"] = method_display(heat_df["xai_method"])
    heat_df["concept_label"] = concept_display(heat_df["concept"])

    heat = pivot_for_heatmap(heat_df, "concept_label", "method", "mean_tixai")
    ordered_concepts = [CONCEPT_LABELS[c] for c in CORE_CONCEPTS if CONCEPT_LABELS[c] in heat.index]
    ordered_methods = [METHOD_LABELS[m] for m in METHOD_ORDER if METHOD_LABELS[m] in heat.columns]
    heat = heat.loc[ordered_concepts, ordered_methods]

    fig, ax = plt.subplots(figsize=(8, 5))
    im = ax.imshow(heat.values, aspect="auto")

    ax.set_xticks(np.arange(len(heat.columns)))
    ax.set_xticklabels(heat.columns)
    ax.set_yticks(np.arange(len(heat.index)))
    ax.set_yticklabels(heat.index)

    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            ax.text(j, i, f"{heat.iloc[i, j]:.3f}", ha="center", va="center", fontsize=8)

    ax.set_title("Pseudo-concept alignment by method: TIxAI")
    fig.colorbar(im, ax=ax, label="Mean TIxAI")
    save_current_fig("fig_pseudo_concept_tixai_heatmap.png")
else:
    print("method_concept_summary is unavailable.")


In [ ]:
# ============================================================
# 5D. Figure: pseudo-concept area ratio
# ============================================================

if method_concept_summary is not None and not method_concept_summary.empty:
    area_df = (
        method_concept_summary
        .groupby("concept", as_index=False)["mean_concept_area"]
        .mean()
    )
    area_df["concept_label"] = concept_display(area_df["concept"])
    area_df = area_df.sort_values("mean_concept_area", ascending=True)

    ax = area_df.plot(
        x="concept_label",
        y="mean_concept_area",
        kind="barh",
        figsize=(8, 5),
        legend=False
    )
    ax.set_title("Mean pseudo-concept area ratio")
    ax.set_xlabel("Mean concept area ratio")
    ax.set_ylabel("Pseudo-concept")
    ax.grid(axis="x", alpha=0.3)

    save_current_fig("fig_pseudo_concept_area_ratio.png")
else:
    print("method_concept_summary is unavailable.")


## 6. Lesion-Size Sensitivity

Purpose: show whether pseudo-concept evaluation is stable across lesion-size categories.

This is especially important for asymmetry and border irregularity, because these pseudo-concepts depend heavily on lesion geometry.


In [ ]:
# ============================================================
# 6A. Figure: lesion-size sensitivity for asymmetry Dice
# ============================================================

if size_summary is not None and not size_summary.empty:
    concept_name = "asymmetry"

    plot_df = size_summary[size_summary["concept"] == concept_name].copy()
    plot_df["method"] = method_display(plot_df["xai_method"])

    available_sizes = [s for s in SIZE_ORDER if s in plot_df["mask_size_class"].unique()]
    pivot = plot_df.pivot_table(
        index="mask_size_class",
        columns="method",
        values="mean_dice",
        aggfunc="mean"
    ).reindex(available_sizes)

    ordered_methods = [METHOD_LABELS[m] for m in METHOD_ORDER if METHOD_LABELS[m] in pivot.columns]
    pivot = pivot[ordered_methods]

    ax = pivot.plot(marker="o", figsize=(9, 5))
    ax.set_title("Lesion-size sensitivity: asymmetry Dice")
    ax.set_xlabel("Mask size class")
    ax.set_ylabel("Mean Dice")
    ax.grid(axis="y", alpha=0.3)

    save_current_fig("fig_size_sensitivity_asymmetry_dice.png")
else:
    print("size_summary is unavailable.")


In [ ]:
# ============================================================
# 6B. Figure: lesion-size sensitivity for colour heterogeneity Dice
# ============================================================

if size_summary is not None and not size_summary.empty:
    concept_name = "colour_heterogeneity"

    plot_df = size_summary[size_summary["concept"] == concept_name].copy()
    plot_df["method"] = method_display(plot_df["xai_method"])

    available_sizes = [s for s in SIZE_ORDER if s in plot_df["mask_size_class"].unique()]
    pivot = plot_df.pivot_table(
        index="mask_size_class",
        columns="method",
        values="mean_dice",
        aggfunc="mean"
    ).reindex(available_sizes)

    ordered_methods = [METHOD_LABELS[m] for m in METHOD_ORDER if METHOD_LABELS[m] in pivot.columns]
    pivot = pivot[ordered_methods]

    ax = pivot.plot(marker="o", figsize=(9, 5))
    ax.set_title("Lesion-size sensitivity: colour heterogeneity Dice")
    ax.set_xlabel("Mask size class")
    ax.set_ylabel("Mean Dice")
    ax.grid(axis="y", alpha=0.3)

    save_current_fig("fig_size_sensitivity_colour_dice.png")
else:
    print("size_summary is unavailable.")


In [ ]:
# ============================================================
# 6C. Figure: Dice vs TIxAI across lesion sizes for asymmetry
# ============================================================

if size_summary is not None and not size_summary.empty:
    concept_name = "asymmetry"

    plot_df = size_summary[size_summary["concept"] == concept_name].copy()
    plot_df["method"] = method_display(plot_df["xai_method"])

    available_sizes = [s for s in SIZE_ORDER if s in plot_df["mask_size_class"].unique()]

    fig, ax = plt.subplots(figsize=(9, 5))

    for method in [METHOD_LABELS[m] for m in METHOD_ORDER]:
        mdf = plot_df[plot_df["method"] == method].set_index("mask_size_class").reindex(available_sizes)
        if not mdf.empty:
            ax.plot(mdf.index, mdf["mean_dice"], marker="o", label=f"{method} Dice")
            ax.plot(mdf.index, mdf["mean_tixai"], marker="s", linestyle="--", label=f"{method} TIxAI")

    ax.set_title("Metric behaviour by lesion size: asymmetry Dice vs TIxAI")
    ax.set_xlabel("Mask size class")
    ax.set_ylabel("Mean metric value")
    ax.grid(axis="y", alpha=0.3)
    ax.legend(ncol=2, fontsize=8)

    save_current_fig("fig_size_sensitivity_asymmetry_dice_vs_tixai.png")
else:
    print("size_summary is unavailable.")


## 7. Label-Stratified Analysis: Melanoma vs Non-Melanoma

Purpose: assess whether XAI alignment with pseudo-concepts differs by diagnostic label.

This section merges the raw metric file with the resolved manifest, using `stem` and `dataset`.

The strongest branch finding was expected around colour heterogeneity and border irregularity.


In [ ]:
# ============================================================
# 7A. Merge metric rows with manifest labels
# ============================================================

label_metrics = None

if metrics_df is not None and manifest is not None and not metrics_df.empty and not manifest.empty:
    label_cols = ["dataset", "stem", "label_name", "mask_size_class", "mask_pixels_224"]
    available_label_cols = [c for c in label_cols if c in manifest.columns]

    label_metrics = metrics_df.merge(
        manifest[available_label_cols].drop_duplicates(),
        on=["dataset", "stem"],
        how="left"
    )

    print(label_metrics.shape)
    display(label_metrics[["dataset", "stem", "xai_method", "concept", "label_name", "dice", "sir", "tixai"]].head())
else:
    print("metrics_df or manifest unavailable.")


In [ ]:
# ============================================================
# 7B. Table: melanoma vs non-melanoma pseudo-concept metrics
# ============================================================

if label_metrics is not None and "label_name" in label_metrics.columns:
    label_table = (
        label_metrics
        .groupby(["label_name", "xai_method", "concept"], as_index=False)
        .agg(
            n=("stem", "nunique"),
            mean_dice=("dice", "mean"),
            mean_iou=("iou", "mean"),
            mean_sir=("sir", "mean"),
            mean_tixai=("tixai", "mean"),
            mean_pearson_corr=("pearson_corr", "mean"),
        )
    )

    label_table["method"] = method_display(label_table["xai_method"])
    label_table["concept_label"] = concept_display(label_table["concept"])

    display(label_table.round(4))
    export_table(label_table.round(6), "table_label_stratified_pseudo_concept_summary.csv")
else:
    print("Label metadata unavailable.")


In [ ]:
# ============================================================
# 7C. Figure: melanoma vs non-melanoma boxplot for colour heterogeneity SIR
# ============================================================

if label_metrics is not None and "label_name" in label_metrics.columns:
    concept_name = "colour_heterogeneity"
    metric_name = "sir"

    plot_df = label_metrics[label_metrics["concept"] == concept_name].copy()
    plot_df = plot_df.dropna(subset=["label_name", metric_name])
    plot_df["method"] = method_display(plot_df["xai_method"])

    methods = [METHOD_LABELS[m] for m in METHOD_ORDER if METHOD_LABELS[m] in plot_df["method"].unique()]
    labels = list(plot_df["label_name"].dropna().unique())

    fig, ax = plt.subplots(figsize=(9, 5))

    positions = []
    data = []
    tick_labels = []
    pos = 1

    for method in methods:
        for label in labels:
            vals = plot_df[(plot_df["method"] == method) & (plot_df["label_name"] == label)][metric_name].values
            data.append(vals)
            positions.append(pos)
            tick_labels.append(f"{method}\n{label}")
            pos += 1
        pos += 0.6

    ax.boxplot(data, positions=positions, widths=0.6, showfliers=False)
    ax.set_xticks(positions)
    ax.set_xticklabels(tick_labels, rotation=45, ha="right")
    ax.set_title("Label-stratified colour heterogeneity alignment: SIR")
    ax.set_ylabel("SIR")
    ax.grid(axis="y", alpha=0.3)

    save_current_fig("fig_label_colour_heterogeneity_sir_boxplot.png")
else:
    print("Label metadata unavailable.")


In [ ]:
# ============================================================
# 7D. Figure: melanoma vs non-melanoma bar chart for selected concepts
# ============================================================

if label_metrics is not None and "label_name" in label_metrics.columns:
    selected_concepts = ["colour_heterogeneity", "border_w16_dil6_sigma10", "asymmetry"]
    metric_name = "sir"

    plot_df = (
        label_metrics[label_metrics["concept"].isin(selected_concepts)]
        .groupby(["label_name", "xai_method", "concept"], as_index=False)[metric_name]
        .mean()
    )
    plot_df["method"] = method_display(plot_df["xai_method"])
    plot_df["concept_label"] = concept_display(plot_df["concept"])

    # Focus on Grad-CAM because the branch summary identified strong melanoma-associated effects there.
    grad_df = plot_df[plot_df["xai_method"] == "gradcam"].copy()
    pivot = grad_df.pivot_table(index="concept_label", columns="label_name", values=metric_name)

    ax = pivot.plot(kind="bar", figsize=(9, 5), rot=30)
    ax.set_title("Grad-CAM pseudo-concept SIR by label")
    ax.set_xlabel("Pseudo-concept")
    ax.set_ylabel("Mean SIR")
    ax.grid(axis="y", alpha=0.3)
    ax.legend(title="Label")

    save_current_fig("fig_gradcam_selected_concepts_sir_by_label.png")
else:
    print("Label metadata unavailable.")


## 8. Chance-Adjusted Pseudo-Concept Evaluation

Purpose: contextualise raw overlap for sparse pseudo-concepts.

This section is optional and runs only if the chance-adjusted CSV files exist in:

`outputs/{RUN_NAME}/metrics/chance_adjusted/`

The key question is:

> Is observed XAI/pseudo-concept overlap higher than expected from random lesion-constrained regions of the same size?


In [ ]:
# ============================================================
# 8A. Inspect chance-adjusted columns
# ============================================================

if chance_metrics is not None and not chance_metrics.empty:
    print(chance_metrics.shape)
    print(chance_metrics.columns.tolist())
    display(chance_metrics.head())
else:
    print("chance_metrics unavailable or empty.")


In [ ]:
# ============================================================
# 8B. Figure: chance-adjusted enrichment by method and concept
# ============================================================

if chance_metrics is not None and not chance_metrics.empty:
    # Try to infer likely enrichment columns.
    possible_enrichment_cols = [
        c for c in chance_metrics.columns
        if "enrichment" in c.lower() or c.lower().endswith("_ratio")
    ]
    print("Candidate enrichment columns:", possible_enrichment_cols)

    # Prefer Dice enrichment if present.
    preferred = None
    for c in possible_enrichment_cols:
        if "dice" in c.lower() and "enrichment" in c.lower():
            preferred = c
            break
    if preferred is None and possible_enrichment_cols:
        preferred = possible_enrichment_cols[0]

    if preferred is None:
        print("No enrichment column found. Please check chance_metrics columns.")
    else:
        plot_df = chance_metrics.copy()
        plot_df["method"] = method_display(plot_df["xai_method"])
        plot_df["concept_label"] = concept_display(plot_df["concept"])

        summary = (
            plot_df
            .groupby(["method", "concept_label"], as_index=False)[preferred]
            .mean()
        )

        heat = summary.pivot(index="concept_label", columns="method", values=preferred)
        ordered_concepts = [CONCEPT_LABELS[c] for c in CORE_CONCEPTS if CONCEPT_LABELS[c] in heat.index]
        ordered_methods = [METHOD_LABELS[m] for m in METHOD_ORDER if METHOD_LABELS[m] in heat.columns]
        heat = heat.loc[ordered_concepts, ordered_methods]

        fig, ax = plt.subplots(figsize=(8, 5))
        im = ax.imshow(heat.values, aspect="auto")

        ax.set_xticks(np.arange(len(heat.columns)))
        ax.set_xticklabels(heat.columns)
        ax.set_yticks(np.arange(len(heat.index)))
        ax.set_yticklabels(heat.index)

        for i in range(heat.shape[0]):
            for j in range(heat.shape[1]):
                ax.text(j, i, f"{heat.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)

        ax.set_title(f"Chance-adjusted pseudo-concept alignment: {preferred}")
        fig.colorbar(im, ax=ax, label=preferred)
        save_current_fig(f"fig_chance_adjusted_heatmap_{preferred}.png")
else:
    print("chance_metrics unavailable or empty.")


In [ ]:
# ============================================================
# 8C. Figure: raw Dice vs chance-adjusted enrichment
# ============================================================

if chance_metrics is not None and not chance_metrics.empty and method_concept_summary is not None:
    # Infer enrichment column.
    possible_enrichment_cols = [
        c for c in chance_metrics.columns
        if "enrichment" in c.lower()
    ]
    preferred = None
    for c in possible_enrichment_cols:
        if "dice" in c.lower():
            preferred = c
            break

    if preferred is None:
        print("No Dice enrichment column found. Available enrichment columns:", possible_enrichment_cols)
    else:
        chance_summary = (
            chance_metrics
            .groupby(["xai_method", "concept"], as_index=False)[preferred]
            .mean()
        )

        merged = method_concept_summary.merge(
            chance_summary,
            on=["xai_method", "concept"],
            how="inner"
        )

        selected = merged[merged["concept"].isin(["asymmetry", "colour_heterogeneity", "border_w16_dil6_sigma10"])].copy()
        selected["method"] = method_display(selected["xai_method"])
        selected["concept_label"] = concept_display(selected["concept"])

        fig, ax1 = plt.subplots(figsize=(10, 5))

        x = np.arange(len(selected))
        width = 0.4

        ax1.bar(x - width/2, selected["mean_dice"], width, label="Raw Dice")
        ax1.set_ylabel("Raw mean Dice")
        ax1.set_xticks(x)
        ax1.set_xticklabels(selected["method"] + "\n" + selected["concept_label"], rotation=45, ha="right")
        ax1.grid(axis="y", alpha=0.3)

        ax2 = ax1.twinx()
        ax2.bar(x + width/2, selected[preferred], width, label=preferred)
        ax2.axhline(1.0, linestyle="--", linewidth=1)
        ax2.set_ylabel("Chance-adjusted enrichment")

        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

        ax1.set_title("Raw Dice vs chance-adjusted enrichment")

        save_current_fig("fig_raw_dice_vs_chance_enrichment.png")
else:
    print("Required chance or method-concept summary unavailable.")


## 9. Automatic Result Notes

This cell prints a short evidence trail that can be copied into the report draft.

Do not copy blindly. Use it as a check that the visual story agrees with the numeric tables.


In [ ]:
# ============================================================
# 9. Automatic result notes
# ============================================================

notes = []

if mask_summary is not None and not mask_summary.empty:
    for metric in ["mean_dice", "mean_iou", "mean_sir", "mean_tixai"]:
        best = mask_summary.loc[mask_summary[metric].idxmax()]
        notes.append(
            f"Lesion-level {metric.replace('mean_', '').upper()}: "
            f"highest for {METHOD_LABELS.get(best['xai_method'], best['xai_method'])} "
            f"({best[metric]:.3f})."
        )

if method_concept_summary is not None and not method_concept_summary.empty:
    concept_best = method_concept_summary.loc[method_concept_summary["mean_dice"].idxmax()]
    notes.append(
        f"Pseudo-concept Dice: highest observed for "
        f"{METHOD_LABELS.get(concept_best['xai_method'], concept_best['xai_method'])} "
        f"on {CONCEPT_LABELS.get(concept_best['concept'], concept_best['concept'])} "
        f"({concept_best['mean_dice']:.3f})."
    )

    area = (
        method_concept_summary
        .groupby("concept")["mean_concept_area"]
        .mean()
        .sort_values()
    )
    smallest = area.index[0]
    largest = area.index[-1]
    notes.append(
        f"Pseudo-concept area: smallest average region is "
        f"{CONCEPT_LABELS.get(smallest, smallest)} ({area.iloc[0]:.4f}); "
        f"largest is {CONCEPT_LABELS.get(largest, largest)} ({area.iloc[-1]:.4f})."
    )

if size_summary is not None and not size_summary.empty:
    asym = size_summary[size_summary["concept"] == "asymmetry"]
    if not asym.empty:
        size_dice = asym.groupby("mask_size_class")["mean_dice"].mean()
        notes.append(
            "Asymmetry size sensitivity: mean Dice by size class = "
            + ", ".join([f"{idx}: {val:.3f}" for idx, val in size_dice.items()])
            + "."
        )

print("\n".join(f"- {n}" for n in notes))


## 10. Suggested Report Captions

Use these as first drafts.

- **Lesion-level metric comparison.** Mean IoU, Dice, SIR and TIxAI scores comparing each XAI map against the lesion segmentation mask. This provides the standard lesion-localisation baseline before pseudo-concept analysis.

- **Pseudo-concept Dice heatmap.** Mean Dice overlap between XAI methods and ABCD-inspired pseudo-concept maps. The heatmap highlights differences between broad lesion localisation and finer concept-level alignment.

- **Pseudo-concept area ratio.** Average spatial size of each pseudo-concept region. Sparse concepts such as border irregularity occupy very small regions, making raw overlap metrics difficult to interpret directly.

- **Lesion-size sensitivity.** Pseudo-concept alignment across lesion-size categories. Declining overlap for small and tiny lesions indicates that pseudo-concept reliability depends strongly on lesion morphology and scale.

- **Dice vs TIxAI by lesion size.** Comparison of geometric overlap and saliency concentration metrics. Divergent behaviour between Dice and TIxAI shows that different metrics reward different localisation properties.

- **Label-stratified pseudo-concept alignment.** Comparison of pseudo-concept alignment between melanoma and non-melanoma lesions. This analysis tests whether XAI methods align more strongly with clinically suspicious structures in melanoma examples.

- **Chance-adjusted enrichment.** Comparison of observed pseudo-concept alignment against lesion-constrained random regions of equal size. This contextualises raw overlap values for sparse pseudo-concept regions.
